<a href="https://colab.research.google.com/github/JTRather/Steam_Pick/blob/main/Steam_Pick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Pick
My app helps Twitch and YouTube creators decide which Steam game to stream next by ranking games based on concurrent players, average playtime, release date, and player ratings.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/


In [3]:

def top_popular(games, n):
    popular = games.sort_values("peak_ccu", ascending=False)
    return popular[["name", "peak_ccu"]].head(n)

print(top_popular(games, 5))


                     name  peak_ccu
0        Counter-Strike 2   1212356
181  Monster Hunter Wilds    703236
1     PUBG: BATTLEGROUNDS    616738
2                  Dota 2    555977
911         Split Fiction    201694


In [4]:

my_function = top_popular
my_result = my_function(games, 5)

print(my_result)


                     name  peak_ccu
0        Counter-Strike 2   1212356
181  Monster Hunter Wilds    703236
1     PUBG: BATTLEGROUNDS    616738
2                  Dota 2    555977
911         Split Fiction    201694


In [5]:

class StreamPickRecommender:

    def __init__(self, games):
        self.games = games.copy()

        # Make sure the columns contain numbers
        self.games["peak_ccu"] = pd.to_numeric(
            self.games["peak_ccu"], errors="coerce"
        )

        self.games["avg_playtime_hours"] = pd.to_numeric(
            self.games["avg_playtime_hours"], errors="coerce"
        )

        self.games["pct_pos_total"] = pd.to_numeric(
            self.games["pct_pos_total"], errors="coerce"
        )

        self.games["num_reviews_total"] = pd.to_numeric(
            self.games["num_reviews_total"], errors="coerce"
        )

    # Question 1: Which games are the most popular?
    def top_popular(self, n=5):

        popular = self.games.sort_values(
            "peak_ccu", ascending=False
        )

        return popular[["name", "peak_ccu"]].head(n)

    # Question 2: Which games have the highest average playtime?
    def top_engaging(self, n=5):

        engaging = self.games[
            self.games["avg_playtime_hours"] > 0
        ]

        engaging = engaging.sort_values(
            "avg_playtime_hours", ascending=False
        )

        return engaging[
            ["name", "avg_playtime_hours"]
        ].head(n)

    # Question 3: Which games combine popularity and good ratings?
    def recommended_games(self, n=5):

        # Remove games with missing information or very few reviews
        recommendations = self.games.dropna(
            subset=[
                "peak_ccu",
                "avg_playtime_hours",
                "pct_pos_total"
            ]
        ).copy()

        recommendations = recommendations[
            (recommendations["avg_playtime_hours"] > 0) &
            (recommendations["num_reviews_total"] >= 100)
        ].copy()

        # Put each measurement on a 0-to-1 scale
        recommendations["popularity_score"] = (
            recommendations["peak_ccu"] /
            recommendations["peak_ccu"].max()
        )

        recommendations["engagement_score"] = (
            recommendations["avg_playtime_hours"] /
            recommendations["avg_playtime_hours"].max()
        )

        recommendations["rating_score"] = (
            recommendations["pct_pos_total"] / 100
        )

        # Calculate an overall recommendation score
        recommendations["recommendation_score"] = (
            recommendations["popularity_score"] * 0.40 +
            recommendations["engagement_score"] * 0.30 +
            recommendations["rating_score"] * 0.30
        )

        recommendations = recommendations.sort_values(
            "recommendation_score", ascending=False
        )

        return recommendations[
            [
                "name",
                "peak_ccu",
                "avg_playtime_hours",
                "pct_pos_total",
                "recommendation_score"
            ]
        ].head(n)


# Create an instance of our class
streampick = StreamPickRecommender(games)

# Display results
print("TOP 5 MOST POPULAR GAMES")
print(streampick.top_popular(5))

print("\nTOP 5 GAMES BY AVERAGE PLAYTIME")
print(streampick.top_engaging(5))

print("\nTOP 5 RECOMMENDED GAMES")
print(streampick.recommended_games(5))


TOP 5 MOST POPULAR GAMES
                     name  peak_ccu
0        Counter-Strike 2   1212356
181  Monster Hunter Wilds    703236
1     PUBG: BATTLEGROUNDS    616738
2                  Dota 2    555977
911         Split Fiction    201694

TOP 5 GAMES BY AVERAGE PLAYTIME
                         name  avg_playtime_hours
254  FINAL FANTASY XIV Online               837.6
2                      Dota 2               717.2
0            Counter-Strike 2               553.2
220           Killing Floor 2               546.2
7                        Rust               353.2

TOP 5 RECOMMENDED GAMES
                         name  peak_ccu  avg_playtime_hours  pct_pos_total  \
0            Counter-Strike 2   1212356               553.2             86   
2                      Dota 2    555977               717.2             81   
254  FINAL FANTASY XIV Online     16445               837.6             87   
220           Killing Floor 2      4315               546.2             88   
7          

In [6]:

import requests
from google.colab import userdata

# Get your credentials from Colab Secrets
client_id = userdata.get("TWITCH_CLIENT_ID")
client_secret = userdata.get("TWITCH_CLIENT_SECRET")

# Ask Twitch for an access token
auth_url = "https://id.twitch.tv/oauth2/token"

auth_data = {
    "client_id": client_id,
    "client_secret": client_secret,
    "grant_type": "client_credentials"
}

response = requests.post(auth_url, data=auth_data)
response.raise_for_status()

# Save the access token
access_token = response.json()["access_token"]

print("Successfully connected to Twitch!")


Successfully connected to Twitch!


In [7]:

# Set up the Twitch API headers
headers = {
    "Client-Id": client_id,
    "Authorization": f"Bearer {access_token}"
}

# Request the top 100 live streams
url = "https://api.twitch.tv/helix/streams"

params = {
    "first": 100
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

# Convert the response into Python data
streams = response.json()["data"]

print("Number of live streams retrieved:", len(streams))

# Display the first five streams
for stream in streams[:5]:
    print(
        stream["user_name"],
        "|",
        stream["game_name"],
        "| Viewers:",
        stream["viewer_count"]
    )


Number of live streams retrieved: 100
IlloJuan | Grand Theft Auto VI | Viewers: 171652
Rubius | Grand Theft Auto VI | Viewers: 107901
alanzoka | Grand Theft Auto VI | Viewers: 82685
auronplay | Grand Theft Auto VI | Viewers: 55887
ohnePixel | Grand Theft Auto VI | Viewers: 54581


In [8]:

import pandas as pd

# Convert Twitch stream data into a DataFrame
twitch_df = pd.DataFrame(streams)

# Group streams by game
twitch_summary = twitch_df.groupby("game_name").agg(
    total_viewers=("viewer_count", "sum"),
    streaming_channels=("user_name", "count"),
    average_viewers=("viewer_count", "mean")
).reset_index()

# Sort by total viewers
twitch_summary = twitch_summary.sort_values(
    "total_viewers",
    ascending=False
)

# Display results
print(twitch_summary.head(10))


              game_name  total_viewers  streaming_channels  average_viewers
3   Grand Theft Auto VI        1467859                  75     19571.453333
4         Just Chatting         161670                  10     16167.000000
5     League of Legends          29721                   4      7430.250000
10    World of Warcraft          17666                   2      8833.000000
7                 Slots          16583                   2      8291.500000
2    Escape from Tarkov          11154                   2      5577.000000
9          Tape to Tape           8871                   1      8871.000000
0                 Chess           8490                   1      8490.000000
1                Dota 2           7314                   1      7314.000000
6             Overwatch           6873                   1      6873.000000


In [9]:

class StreamPickRecommender:

    def __init__(self, games):
        self.games = games.copy()

        # Make sure the columns contain numbers
        self.games["peak_ccu"] = pd.to_numeric(
            self.games["peak_ccu"], errors="coerce"
        )

        self.games["avg_playtime_hours"] = pd.to_numeric(
            self.games["avg_playtime_hours"], errors="coerce"
        )

        self.games["pct_pos_total"] = pd.to_numeric(
            self.games["pct_pos_total"], errors="coerce"
        )

        self.games["num_reviews_total"] = pd.to_numeric(
            self.games["num_reviews_total"], errors="coerce"
        )

    # Question 1: Which games are the most popular?
    def top_popular(self, n=5):

        popular = self.games.sort_values(
            "peak_ccu", ascending=False
        )

        return popular[["name", "peak_ccu"]].head(n)

    # Question 2: Which games have the highest average playtime?
    def top_engaging(self, n=5):

        engaging = self.games[
            self.games["avg_playtime_hours"] > 0
        ]

        engaging = engaging.sort_values(
            "avg_playtime_hours", ascending=False
        )

        return engaging[
            ["name", "avg_playtime_hours"]
        ].head(n)

    # Question 3: Which games combine popularity and good ratings?
    def recommended_games(self, n=5):

        # Remove games with missing information or very few reviews
        recommendations = self.games.dropna(
            subset=[
                "peak_ccu",
                "avg_playtime_hours",
                "pct_pos_total"
            ]
        ).copy()

        recommendations = recommendations[
            (recommendations["avg_playtime_hours"] > 0) &
            (recommendations["num_reviews_total"] >= 100)
        ].copy()

        # Put each measurement on a 0-to-1 scale
        recommendations["popularity_score"] = (
            recommendations["peak_ccu"] /
            recommendations["peak_ccu"].max()
        )

        recommendations["engagement_score"] = (
            recommendations["avg_playtime_hours"] /
            recommendations["avg_playtime_hours"].max()
        )

        recommendations["rating_score"] = (
            recommendations["pct_pos_total"] / 100
        )

        # Calculate an overall recommendation score
        recommendations["recommendation_score"] = (
            recommendations["popularity_score"] * 0.40 +
            recommendations["engagement_score"] * 0.30 +
            recommendations["rating_score"] * 0.30
        )

        recommendations = recommendations.sort_values(
            "recommendation_score", ascending=False
        )

        return recommendations[
            [
                "name",
                "peak_ccu",
                "avg_playtime_hours",
                "pct_pos_total",
                "recommendation_score"
            ]
        ].head(n)


# Create an instance of our class
streampick = StreamPickRecommender(games)

# Display results
print("TOP 5 MOST POPULAR GAMES")
print(streampick.top_popular(5))

print("\nTOP 5 GAMES BY AVERAGE PLAYTIME")
print(streampick.top_engaging(5))

print("\nTOP 5 RECOMMENDED GAMES")
print(streampick.recommended_games(5))


TOP 5 MOST POPULAR GAMES
                     name  peak_ccu
0        Counter-Strike 2   1212356
181  Monster Hunter Wilds    703236
1     PUBG: BATTLEGROUNDS    616738
2                  Dota 2    555977
911         Split Fiction    201694

TOP 5 GAMES BY AVERAGE PLAYTIME
                         name  avg_playtime_hours
254  FINAL FANTASY XIV Online               837.6
2                      Dota 2               717.2
0            Counter-Strike 2               553.2
220           Killing Floor 2               546.2
7                        Rust               353.2

TOP 5 RECOMMENDED GAMES
                         name  peak_ccu  avg_playtime_hours  pct_pos_total  \
0            Counter-Strike 2   1212356               553.2             86   
2                      Dota 2    555977               717.2             81   
254  FINAL FANTASY XIV Online     16445               837.6             87   
220           Killing Floor 2      4315               546.2             88   
7          